In [3]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/redwankarimsony/heart-disease-data/heart_disease_uci.csv")
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


**Week5**

Implement the MLP using the Types of Regularization Techniques,
L2 Regularization ,
Dataset Augmentation ,
Parameter sharing and tying,
Adding noise to the inputs and outputs ,
Early stopping,
Ensemble methods,
Dropouts.


In [4]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/redwankarimsony/heart-disease-data/heart_disease_uci.csv")

# Drop unnecessary
df = df.drop(["id", "dataset"], axis=1)

# Convert target to binary
df["num"] = df["num"].apply(lambda x: 1 if x > 0 else 0)

# Convert categorical → numeric
df = pd.get_dummies(df, drop_first=True)

# Fill missing values
df = df.fillna(df.mean())

df.head()

,age,trestbps,chol,thalch,oldpeak,ca,num,sex_Male,cp_atypical angina,cp_non-anginal,cp_typical angina,fbs_True,restecg_normal,restecg_st-t abnormality,exang_True,slope_flat,slope_upsloping,thal_normal,thal_reversable defect
0,63,145.0,233.0,150.0,2.3,0.0,0,True,False,False,True,True,False,False,False,False,False,False,False
1,67,160.0,286.0,108.0,1.5,3.0,1,True,False,False,False,False,False,False,True,True,False,True,False
2,67,120.0,229.0,129.0,2.6,2.0,1,True,False,False,False,False,False,False,True,True,False,False,True
3,37,130.0,250.0,187.0,3.5,0.0,0,True,False,True,False,False,True,False,False,False,False,True,False
4,41,130.0,204.0,172.0,1.4,0.0,0,False,True,False,False,False,False,False,False,False,True,True,False


In [5]:
from sklearn.model_selection import train_test_split

X = df.drop("num", axis=1)
y = df["num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from sklearn.metrics import accuracy_score
import numpy as np

#  Convert to numpy
X_train_np = np.array(X_train_scaled)
X_test_np = np.array(X_test_scaled)

y_train_np = np.array(y_train)
y_test_np = np.array(y_test)

#  Base function
def evaluate_model(model, name):
    model.fit(X_train_np, y_train_np, epochs=50, verbose=0)
    pred = (model.predict(X_test_np) > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test_np, pred)
    print(name, "-> Accuracy:", acc)

#  1. L2 Regularization
model = Sequential([
    Input(shape=(X_train_np.shape[1],)),
    Dense(30, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    Dense(15, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy')
evaluate_model(model, "L2")

#  2. Dropout
model = Sequential([
    Input(shape=(X_train_np.shape[1],)),
    Dense(30, activation='relu'),
    Dropout(0.3),
    Dense(15, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy')
evaluate_model(model, "Dropout")

#  3. Early Stopping
early = EarlyStopping(monitor='loss', patience=5)

model = Sequential([
    Input(shape=(X_train_np.shape[1],)),
    Dense(30, activation='relu'),
    Dense(15, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy')

model.fit(X_train_np, y_train_np, epochs=100, callbacks=[early], verbose=0)
pred = (model.predict(X_test_np) > 0.5).astype(int).flatten()
print("EarlyStopping -> Accuracy:", accuracy_score(y_test_np, pred))

#  4. Adding Noise
X_train_noisy = X_train_np + 0.1 * np.random.randn(*X_train_np.shape)

model = Sequential([
    Input(shape=(X_train_np.shape[1],)),
    Dense(30, activation='relu'),
    Dense(15, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy')
model.fit(X_train_noisy, y_train_np, epochs=50, verbose=0)

pred = (model.predict(X_test_np) > 0.5).astype(int).flatten()
print("Noise -> Accuracy:", accuracy_score(y_test_np, pred))

#  5. Ensemble
preds = []

for i in range(3):
    model = Sequential([
        Input(shape=(X_train_np.shape[1],)),
        Dense(30, activation='relu'),
        Dense(15, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.fit(X_train_np, y_train_np, epochs=50, verbose=0)

    preds.append((model.predict(X_test_np) > 0.5).astype(int))

final_pred = np.round(np.mean(preds, axis=0)).flatten()
print("Ensemble -> Accuracy:", accuracy_score(y_test_np, final_pred))

2026-04-12 12:04:40.526929: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
L2 -> Accuracy: 0.8478260869565217
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Dropout -> Accuracy: 0.8152173913043478
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
EarlyStopping -> Accuracy: 0.8260869565217391
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Noise -> Accuracy: 0.8260869565217391
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Ensemble -> Accuracy: 0.875


Data Augumentation and Parameter sharing

In [11]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# 🔹 Load images
base_path = "/kaggle/input/datasets/xiaose/cityscapes/Cityspaces/images/val"

cities = os.listdir(base_path)

X = []
y = []

for label, city in enumerate(cities):
    folder = os.path.join(base_path, city)

    for file in os.listdir(folder):
        if file.endswith(".png"):
            img = cv2.imread(os.path.join(folder, file))
            img = cv2.resize(img, (64,64))

            X.append(img)
            y.append(label)

X = np.array(X) / 255.0
y = np.array(y)

print("Data shape:", X.shape)

# 🔹 Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 🔹 Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

datagen.fit(X_train)

# 🔹 CNN Model (Parameter Sharing happens here)
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),   # Regularization
    Dense(3, activation='softmax')   # 3 classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 🔹 Early Stopping
early_stop = EarlyStopping(monitor='loss', patience=3)

# 🔹 Train
model.fit(datagen.flow(X_train, y_train, batch_size=32),
          epochs=10,
          callbacks=[early_stop])

# 🔹 Evaluate
loss, acc = model.evaluate(X_test, y_test)

print("Final Accuracy:", acc)

Data shape: (500, 64, 64, 3)
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.4936 - loss: 1.0293
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.5182 - loss: 0.8888
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.6049 - loss: 0.7534
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 101ms/step - accuracy: 0.7059 - loss: 0.7072
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.7061 - loss: 0.6814
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step - accuracy: 0.6778 - loss: 0.7217
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - accuracy: 0.7098 - loss: 0.6609
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.7910 - loss: 0.5383
Epoch 9/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.7829 - loss: 0.5621
Epoch 10/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step - accuracy: 0.8377 - loss: 0.4849
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8339 - loss: 0.3574
Final Accuracy: 0.8399999737739563


Observation
* The CNN model successfully learned spatial features from the Cityscapes dataset and showed steady improvement in accuracy during training.
* Dataset augmentation and dropout helped improve generalization and reduce overfitting.
* The model achieved good accuracy despite the complexity of real-world image data, demonstrating the effectiveness of CNN for image classification tasks.